<img src="image01.png" alt="Project image" style="display: block; margin: 0 auto;">

<img src="Logo-svg.png" alt="university image" style="width: 400px; display: block; margin: 0 auto;">

# CS616 Project: Solving TSP using Hybrid Simulated Annealing + k-opt

**Course:** CS616 – Optimization Algorithms  
**Semester:** Fall 2025/2026  
**Student:** Mohammed Mizan Alanazi (446540203)  
**Supervisor:** Dr. Mahdi Khemakhem  
**Institution:** Prince Sattam bin Abdulaziz University

---

## Project Overview

This notebook implements a hybrid metaheuristic algorithm combining:
1. **Simulated Annealing (SA)** - Probabilistic global search framework
2. **k-opt Local Search** - Deterministic local refinement (2-opt Reverse sub-routes)
3. **Reheating Mechanism** - Temperature reset to escape stagnation
4. **Perturbation Strategy** - Swap random sub-routes for diversification

The algorithm is evaluated on **10 TSPLIB benchmark instances** ranging from 51 to 1002 cities.

---

## 1. Problem Description:

The **Traveling Salesperson Problem (TSP)** requires finding a route, or tour, that visits a set of $n$ cities exactly once and returns to the starting city, such that the **total distance (or cost) traveled is minimized**.

The problem is defined on a set of nodes (cities) and edges (roads) where each edge has an associated cost. The solution must be a **Hamiltonian Cycle**, which is a cycle that passes through every node exactly once.

The TSP is a foundational problem in combinatorial optimization, often used to model and solve real-world problems such as:
* **Logistics and Transportation:** Route planning for delivery vehicles, mail carriers, or school buses.
* **Manufacturing:** Drilling paths for circuit boards (NC drilling) or controlling robotic assembly lines.
* **Bioinformatics:** DNA sequencing (ordering fragments into a complete sequence).
* **Microchip Design:** Wire routing and placement optimization.

The global decision of the TSP is to determine the optimal sequence of cities to visit to minimize the total travel cost.

The **symmetric TSP** assumes the cost of travel between two cities is the same in both directions ($d_{ij} = d_{ji}$), while the formulation below addresses the **asymmetric TSP**, where $d_{ij}$ may not equal $d_{ji}$.

The TSP is **NP-hard**. It is one of the most studied problems in optimization because it is easy to state but incredibly difficult to solve exactly for large instances.


---
## 🗺️  Explanation of the TSP Problem

## 2. Mathematical Formulation (Asymmetric TSP)

To model the Traveling Salesperson Problem, we define the following components:

#### Sets:
* $N = \{1, 2, \ldots, n\}$: Set of $n$ cities (nodes).

#### Parameters:
* $d_{ij}$: Cost (distance) of traveling directly from city $i \in N$ to city $j \in N$ ($i \neq j$).

#### Decision Variables:
$$\forall i, j \in N, i \neq j, x_{ij} =
\begin{cases}
1 & \text{ if the tour includes the edge from city } i \text{ to city } j \\
0 & \text{ otherwise}
\end{cases}
$$

#### Objective Function:

Minimize the total distance/cost of the tour:

$$\text{Minimize} \quad f(x) = \sum_{i=1}^{n} \sum_{\substack{j=1 \\ j \neq i}}^{n} d_{ij} x_{ij}$$

#### Constraints:
1.  **Out-degree Constraint (Leave each city once):** Each city must be left exactly once.
    $$\sum_{\substack{j=1 \\ j \neq i}}^{n} x_{ij} = 1, \quad \forall i \in N$$

2.  **In-degree Constraint (Enter each city once):** Each city must be entered exactly once.
    $$\sum_{\substack{i=1 \\ i \neq j}}^{n} x_{ij} = 1, \quad \forall j \in N$$

3.  **Subtour Elimination Constraint (SEC):** Prevents the formation of two or more disconnected loops (subtours) by ensuring that the number of selected edges within any proper subset of cities $Q$ is less than the number of cities in $Q$.
    $$\sum_{i \in Q} \sum_{\substack{j \in Q \\ j \neq i}} x_{ij} \le |Q| - 1, \quad \forall Q \subsetneq N, |Q| \ge 2$$

4.  **Binary Constraints** on decision variables:
    $$x_{ij} \in \{0, 1\}, \quad \forall i, j \in N, i \neq j$$



## 3. Dataset Description

This project evaluates the hybrid SA + k-opt algorithm on **10 TSPLIB benchmark instances** spanning three complexity tiers[1]:

### Dataset Composition

| Instance Name | Number of Cities | Scale | Characteristics |
|---|---|---|---|
| eil51.tsp | 51 | Small | Clustered Euclidean cities |
| berlin52.tsp | 52 | Small | Berlin city coordinates |
| pr299.tsp | 299 | Medium | Geometric Euclidean instance |
| lin318.tsp | 318 | Medium | Large Euclidean problem |
| rd400.tsp | 400 | Medium-Large | Random Euclidean distribution |
| pr439.tsp | 439 | Medium-Large | Geometric pattern |
| rat575.tsp | 575 | Large | Random Euclidean instance |
| d657.tsp | 657 | Large | Geometric structure |
| rat783.tsp | 783 | Large | Dense random distribution |
| pr1002.tsp | 1002 | Very Large | Largest benchmark instance |

### Dataset Categories

**Small-scale instances (51–52 cities):** These instances enable verification of algorithm correctness and serve as baseline comparisons. Solutions can be found quickly, allowing for parameter tuning and initial feasibility testing.

**Medium-scale instances (299–439 cities):** These instances represent the transition zone where exact methods begin to struggle, making metaheuristics necessary. They provide practical insight into algorithm performance on realistic problems.

**Large-scale instances (575–1002 cities):** These instances challenge the algorithm's scalability and convergence properties. The largest instance (pr1002) represents a significant computational challenge, requiring efficient implementation and good parameter tuning to find competitive solutions within reasonable time.

### TSPLIB Repository

All instances are obtained from the **TSPLIB repository**, a standardized collection of TSP benchmark problems maintained for research purposes[1]. The repository includes:

* **Known optimal or best-known solutions (BKS)** for most instances, enabling accurate optimality gap calculation
* **Diverse instance types:** Euclidean TSP (cities as points in 2D plane), geometric patterns, and random distributions
* **Consistent file format** enabling standardized benchmark comparison across algorithms

The Python `tsplib95` library is used to parse and load these instances.

***

## 4. Proposed Algorithm: Hybrid Simulated Annealing + k-opt

Rather than solving the TSP formulation exactly (which is computationally infeasible for large **n**), this project develops a **hybrid metaheuristic approach** that combines Simulated Annealing with k-opt local search to obtain high-quality approximate solutions efficiently.

### 4.1 Algorithm Components

**Simulated Annealing (SA):** SA is a probabilistic global search framework inspired by the physical annealing process in metallurgy, where a material is slowly cooled to achieve a low-energy crystalline state. In optimization, this translates to a search mechanism that probabilistically accepts worse solutions early in the search (high temperature) to escape local minima, gradually becoming more selective as the temperature decreases.

**k-opt Local Search:**  
Local search operators iteratively improve tours by removing **k** edges and reconnecting them in a different configuration. The **2-opt** operator removes 2 edges.  
This operator provide fine-grained refinement of candidate solutions.

**Reheating Mechanism:** When the algorithm detects stagnation (no improvement over iterations), the temperature is reset to enable further exploration. Multiple reheating cycles allow the algorithm to escape different local optima.

**Perturbation Strategy:** Adaptive controlled randomness is introduced to diversify the search space beyond what local search alone can achieve.

***

### 4.2 Algorithm Formulation

#### **Metropolis Acceptance Criterion** 

The acceptance probability for moving from current solution $$s$$ to candidate solution $$s'$$ is:

$$P(\text{accept } s' | s, T) = \begin{cases}
1 & \text{if } f(s') \leq f(s) \\
\exp\left(\frac{f(s) - f(s')}{T}\right) & \text{if } f(s') > f(s)
\end{cases}$$

where $T$ is the current temperature and $f(\cdot)$ is the tour cost.\
\
This criterion ensures that improving solutions are always accepted, while worse solutions are accepted with probability that decreases exponentially with the cost increase and temperature.

#### **Temperature Schedule – Exponential Cooling** 

$$T_k = T_0 \cdot \alpha^k \quad \text{where} \quad \alpha \in (0, 1)$$

The cooling rate $\alpha$ controls the speed of temperature decrease. Slower cooling (higher $\alpha$) generally yields better solutions at the cost of longer computation time.

#### **Initial Temperature Selection** 

$$T_0 = C \cdot \overline{\Delta E} \quad \text{or} \quad T_0 \geq \max_{\text{observed}} |\Delta E|$$

The initial temperature is set proportionally to the average cost difference $\overline{\Delta E}$ observed during initial exploration, ensuring that most initial moves are accepted.

#### **Markov Chain Length** 

$$M = \beta \cdot n$$

The number of moves attempted at each temperature is scaled linearly with the problem size $n$, where $\beta$ is the Markov chain multiplier. Larger problems receive proportionally more iterations at each temperature level.

#### **Reheating Strategy – Progressive Reset** 

$$T^{(i)} = \frac{T_0}{\sqrt{i}}, \quad i = 1, 2, \ldots, N_{\text{reheat}}$$

After stagnation is detected, the temperature is reset to enable additional exploration phases. The strength of each reheating cycle is progressively reduced.

#### **2-opt Move Evaluation** 

$$\Delta d = d(c_i, c_j) + d(c_{i+1}, c_{j+1}) - d(c_i, c_{i+1}) - d(c_j, c_{j+1})$$

This computes the cost change from swapping edges in the tour. If $\Delta d < 0$, the move improves the tour.

#### **k-opt Neighborhood Size** 

$$|N_k(s)| = \mathcal{O}(n^k)$$

The size of the neighborhood structure grows polynomially with problem size and the number of edges replaced. This represents the search space complexity at each step.

#### **Adaptive Perturbation Strength** 

$$k_{\text{pert}} = k_{\min} + \left\lfloor (k_{\max} - k_{\min}) \cdot \frac{T}{T_0} \right\rfloor$$

The perturbation strength decreases as temperature drops, allowing aggressive diversification early and fine-grained refinement later.

#### **Stopping Criteria** 

Stop if:
$$T \leq T_{\min} \quad \text{or} \quad \text{iterations} \geq K_{\max}$$

The algorithm terminates when the temperature falls below a minimum threshold or the iteration limit is reached.

***

## 5. Implementation Details

The algorithm is implemented in **Python** using the following libraries:

* **NumPy:** For efficient numerical computations and matrix operations
* **Pandas:** For data manipulation, aggregation, and results export
* **TSPLIB95:** For loading and parsing standard TSP benchmark instances
* **Matplotlib & Seaborn:** For generating convergence plots, trajectory analysis, and statistical visualizations
* **Numba:** **(New Add)** For JIT compilation to accelerate computationally intensive functions (e.g., 2-opt local search)
* **Joblib:** For parallel execution of experiments across multiple CPU cores
* **Gurobi (Optional):** For obtaining exact optimal solutions to use as a baseline

### 5.1 Algorithm Initialization

* **Initial Solution:** Generated using a **greedy nearest-neighbor heuristic** to provide a reasonable starting point
* **Distance Matrix:** Precomputed Euclidean distances between all pairs of cities for efficient evaluation
* **Temperature:** Set based on observed cost differences during initial exploration

### 5.2 Parameter Configuration

| Parameter | Value | Description |
|---|---|---|
| $$T_0$$ | $$0.1 \cdot \overline{d}$$ | Initial temperature controlling exploration intensity |
| $$T_{\min}$$ | $$10^{-3} T_0$$ | Minimum temperature; algorithm stopping limit |
| $$\alpha$$ | 0.95–0.99 | Cooling rate; slower rate yields higher-quality solutions |
| $$\beta$$ | 10–100 | Markov chain multiplier determining iteration length per temperature |
| $$k_{\min}$$ | 2 | Minimum perturbation strength (2-opt neighborhood) |
| $$k_{\max}$$ | 5 | Maximum perturbation strength (up to 5-opt) |
| $$K_{\max}$$ | 10³–10⁴ | Maximum number of outer iterations; scales with $n$ |
| $$N_{\text{reheat}}$$ | 3–5 | Number of reheating cycles for diversification |

***

## 6. Evaluation Metrics

The proposed algorithm will be evaluated using the following performance metrics:

* **Tour Length (Total Distance):** The objective function value of the best solution found
* **Optimality Gap (%):** Percentage deviation from known best-known solutions: $$\text{Gap} = \frac{\text{Tour Length} - \text{BKS}}{\text{BKS}} \times 100\%$$
* **Computational Time (CPU Time):** Total execution time in seconds
* **Convergence Behavior:** Solution quality over iterations (convergence plots)
* **Solution Stability:** Standard deviation of solution quality across multiple runs
* **Speedup Analysis:** Comparison of computational efficiency against exact solvers and competing metaheuristics

***

## 7. Baseline Methods for Comparison

The proposed hybrid algorithm will be benchmarked against:

1. **Nearest Neighbor (NN):** Greedy construction heuristic baseline
2. **Pure Simulated Annealing (SA):** SA without k-opt refinement
3. **Local Search (k-opt only):** Deterministic local search without global exploration
4. **SA + k-opt (Proposed):** The hybrid algorithm combining both techniques
5. **Gurobi (Exact Solver):** Commercial MIP solver for solution quality bounds

***

## 8. Expected Contributions

The expected contributions of this project include:

* **Design of a hybrid optimization approach** that combines probabilistic global exploration (Simulated Annealing) with focused local refinement (k-opt moves), enabling more effective navigation of complex solution spaces
* **Empirical analysis** demonstrating when and why hybrid approaches outperform pure global or pure local search strategies
* **Systematic evaluation** on diverse problem instances (51 to 1,002 cities) providing insights into scalability and parameter sensitivity
* **Practical implementation insights** for practitioners seeking to apply metaheuristic optimization to real-world routing and logistics problems

***




## Import Library 📚

In [4]:
import os
import time
import gc
import io
import sys
import contextlib
import math
import warnings
import numpy as np
import pandas as pd
from numba import njit
import tsplib95
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
from pathlib import Path
from joblib import Parallel, delayed
from multiprocessing import Manager
try:
    import gurobipy as gp
    from gurobipy import GRB
    GUROBI_AVAILABLE = True
except ImportError:
    GUROBI_AVAILABLE = False
    print("⚠️ Gurobi not found. Gurobi baseline will be skipped.")

print("Libraries Imported! ✅")

Libraries Imported! ✅


# ⚙️ Configuration & Environment Setup

Establishes the global environment for the experiments:
- **Directories**: Sets up paths for data, results, figures, and trajectories.
- **Parameters**: Defines global defaults (seed, cooling rate $\alpha$, Markov chain length $\beta$).
- **Instances**: Lists the 10 specific TSPLIB instances to be tested.
- **Environment**: Seeds the random number generator for reproducibility.

In [5]:
CONFIG = {
    'dirs': {
        'data': 'data/tsplib_instances/',
        'results': 'results/',
        'figures': 'results/figures/',

    },
    'params': {
        'seed': 42,
        'T0_factor': 0.1,
        'Tmin_factor': 1e-3,
        'alpha': 0.98,
        'beta': 10,
        'n_reheat': 3,
        'perturb_freq': 100,
        'k_min': 2,
        'k_max': 5,
        
        
        'time_limit': 600.0,       
        'pure_sa_timelimit': 60.0, 
        'max_iterations': 15000,
        
        'stagnation_limit_small': 100,
        'stagnation_limit_med': 200,
        'stagnation_limit_large': 300,
        
        'perturb_strength_weak': 4,
        'perturb_strength_strong': 6,
        'perturb_strength_restart': 8,
        
        'max_restarts_without_improvement': 5,
        
        # Gurobi Settings
        'gurobi_timelimit': 300.0,
        'gurobi_gap': 0.00
    },
    'instances': [
        'eil51.tsp', 'berlin52.tsp', 'pr299.tsp', 'lin318.tsp', 
        'rd400.tsp', 'pr439.tsp', 'rat575.tsp', 'd657.tsp', 
        'rat783.tsp', 'pr1002.tsp'
    ]
}

# Setup
np.random.seed(CONFIG['params']['seed'])
for d in CONFIG['dirs'].values():
    Path(d).mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
print("Configuration Ready! ✅")

Configuration Ready! ✅


# 🛠️ Utility Helper Functions

Collection of helper functions for data processing:
- **format_time()**: Converts execution seconds into a human-readable format (e.g., "1m 30s").
- **calculate_gap()**: Computes the percentage deviation from the known optimal solution.
- **get_instance_params()**: Dynamically adjusts hyperparameters (like `max_iterations` and `time_limit`) based on the problem size to ensure fair comparisons.

In [6]:
def format_time(seconds):
    if seconds < 60: return f"{seconds:.2f}s"
    mins, secs = divmod(seconds, 60)
    if mins < 60: return f"{int(mins)}m {int(secs)}s"
    return f"{int(mins//60)}h {int(mins%60)}m"

def calculate_gap(obtained, optimal):
    if not optimal: return 0.0
    return ((obtained - optimal) / optimal) * 100

def get_instance_params(size):
    p = CONFIG['params'].copy()
    if size > 800:
        p.update({'beta': 10, 'alpha': 0.98, 'time_limit': 120.0, 'max_iterations': 15000})
    elif size > 500:
        p.update({'beta': 10, 'alpha': 0.98, 'time_limit': 90.0, 'max_iterations': 10000})
    elif size > 250:
        p.update({'beta': 10, 'alpha': 0.999, 'time_limit': 60.0, 'max_iterations': 5000})
    return p

print("Utilities Ready! ✅")

Utilities Ready! ✅


# 📦 TSP Instance Loader

Handles loading and processing of TSPLIB problem files:
- **Loading**: Parses `.tsp` files using `tsplib95`.
- **Distance Matrix**: Pre-computes the $N \times N$ distance matrix for fast $O(1)$ lookups.
- **Candidate Lists**: Pre-computes nearest neighbors to speed up local search.
- **Optimal Lookup**: Retrieves known optimal values for gap calculation.

In [7]:
class TSPInstance:
    def __init__(self, filepath):
        self.problem = tsplib95.load(filepath)
        self.name = self.problem.name
        self.dimension = self.problem.dimension
        self.optimal = self._get_optimal(self.name)
        self.matrix = self._build_matrix()
        self.candidates = self._build_candidates(k=20)
        self.matrix_mean = self.matrix.mean() 


    def _build_matrix(self):
        n = self.dimension
        mat = np.zeros((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                w = self.problem.get_weight(i + 1, j + 1)
                mat[i, j] = mat[j, i] = w
        return mat
    
    def _build_candidates(self, k=20):
        return np.argsort(self.matrix, axis=1)[:, 1:k+1]

    def _get_optimal(self, name):
        optima = {
            'eil51': 426, 'berlin52': 7542, 'pr299': 48191, 
            'lin318': 42029, 'rd400': 15281, 'pr439': 107217,
            'rat575': 6773, 'd657': 48912, 'rat783': 8806, 'pr1002': 259045
        }
        return optima.get(name, 0)

    def evaluate(self, tour):
        tour = np.asarray(tour, dtype = np.int64)
        return self.matrix[tour, np.roll(tour, -1)].sum()

print("TSPInstance class defined successfully! ✅")

TSPInstance class defined successfully! ✅


# 🧬 Solution Encoding (Permutation Representation)

Represents a TSP solution using permutation encoding:
- **tour**: Ordered list of city indices (e.g., `[0, 2, 4, 1, 3]`).
- **total_dist**: Tour length (total distance).
- **Methods**:
  - `is_valid()`: checks for duplicate or missing cities.
  - `copy()`: creates a deep copy to prevent reference errors during mutation.

In [8]:
class TSPSolution:
    def __init__(self, tour, instance):
        self.tour = np.array(tour, dtype= int)
        self.instance = instance
        self.total_dist = instance.evaluate(self.tour)

    def copy(self):
        return TSPSolution(self.tour.copy(), self.instance)
    
    def is_valid(self):
        return (len(self.tour) == self.instance.dimension and
                len(set(self.tour)) == self.instance.dimension)
    
    print("TSPSolution class defined successfully! ✅")

TSPSolution class defined successfully! ✅


# 🏃 Nearest Neighbor Heuristic

A greedy constructive heuristic used to generate initial solutions:
- **Logic**: Starts at a random city and repeatedly visits the nearest unvisited neighbor.
- **Purpose**: Provides a high-quality starting point for Simulated Annealing, significantly faster than random initialization.
- **Time Complexity**: $O(N^2)$, where $N$ is the number of cities (calculates distances to all unvisited nodes at each step).

In [9]:
def nearest_neighbor(instance):
    n = instance.dimension
    start = np.random.randint(0,n)
    unvisited = set(range(n))
    unvisited.remove(start)
    tour = [start]
    curr = start
    while unvisited:
        nxt = min(unvisited, key= lambda x: instance.matrix[curr][x])
        tour.append(nxt)
        unvisited.remove(nxt)
        curr = nxt
    return TSPSolution


print("Nearest Neighbor Ready! ✅")

Nearest Neighbor Ready! ✅


# 🔍 Accelerated 2-opt Local Search

A deterministic local search operator optimized with **Numba**:
- **Logic**: Iteratively swaps edges to uncross lines in the tour (reverses segments) until no further improvements can be made (Local Optimum).
- **Optimization**: Uses candidate lists (nearest $k$ neighbors) to reduce the search space from $O(N^2)$ to $O(N \cdot k)$ per pass.
- **Time Complexity**: Approximately $O(N)$ per pass (with candidate lists) or $O(N^2)$ in the standard case.

In [ ]:
@njit
def two_opt_boosted(tour, mat, candidates):
    n = len(tour)
    improved = True
    pos = np.zeros(n, dtype= np.int64)
    pos[tour] = np.arange(n)

    while improved:
        improved = False
        for i in range(n -1):
            u = tour[i]
            u_next = tour[i + 1]
            for v in candidates[u]:
                j = pos[v]
                if j <= i + 1: continue
                v_next = tour[(j + 1) % n]

                if mat[u][v] + mat[u_next][v_next] < mat[u][u_next] + mat[v][v_next]:
                    tour[i+1 : j+1] = tour[i+1 : j+1][::-1]
                    pos[tour[i+1 : j+1]] = np.arange(i+1, j+1)
                    improved = True
                    break
            if improved:
                break
    return tour

def two_opt(solution):
    tour = solution.tour.copy().astype(np.int64)
    mat = solution.instance.matrix
    candidates = solution.instance.candidates
    new_tour = two_opt_boosted(tour, mat, candidates)    
    solution.tour = new_tour
    solution.total_dist = solution.instance.evaluate(new_tour)
    return solution

print("Two Opt Ready! ✅")

Two Opt Ready! ✅


# 🎲 SA Primitives & Perturbation

Core mechanisms for the Simulated Annealing process:
- **generate_neighbor**: Creates a neighbor using a random swap.
- **perturb_solution**: Applies a non-deterministic (**Random**) "swap" (multiple swaps) to escape local optima during stagnation.
- **accept_criterion**: The Metropolis criterion ($P = e^{-\Delta/T}$) to determine if a worsening move should be accepted.

In [ ]:
@njit
def generate_neighbor(solution):
    n = len(solution.tour)
    i, j = sorted(np.random.choice(n, 2, replace= False))
    new_tour = solution.tour.copy()
    new_tour[i:j+1] = new_tour[i:j+1][::-1]
    return TSPSolution(new_tour, solution.instance)

def perturb_solution(solution, strength):
    new_sol = solution.copy()
    n = len(new_sol.tour)
    for _ in range (strength):
        i, j = np.random.choice(n, 2, replace= False)
        new_sol.tour[i], new_sol.tour[j] = new_sol.tour[j], new_sol.tour[i]
    new_sol.total_dist = new_sol.instance.evaluate(new_sol.tour)
    return new_sol

def accept_criterion(delta, T):
    return delta < 0 or np.random.random() < np.exp(-delta/ T)

print("SA Primitives Ready! ✅")

SA Primitives Ready! ✅


# 🔥 Enhanced Simulated Annealing (ESA)

The core hybrid metaheuristic algorithm. It combines:
1.  **Simulated Annealing**: Geometric cooling schedule to explore the search space.
2.  **Local Search**: Applies 2-opt improvements to refining candidate solutions.
3.  **Reheating**: Detects stagnation and "reheats" the temperature to escape deep local optima.
4.  **History Tracking**: Records convergence data, acceptance rates, and events for visualization.

In [14]:
def solve_enhanced_sa(instance, params, track = False):
    history = {
        'total_dist': [],
        'current_dist': [],
        'temp': [],
        'accepted': [],
        'reheat_iters': []
    }

    curr = two_opt(nearest_neighbor(instance))
    best = curr.copy()
    T = params['T0_factor'] * instance.matrix_mean
    alpha = params['alpha']
    max_time = params['time_limit']
    max_iter = params.get('max_iterations', 15000)


    if instance.dimension > 500:
        max_stagnant = params.get('stagnation_limit_large', 300)
    elif instance.dimension > 250:
        max_stagnant = params.get('stagnation_limit_med', 200)
    else:
        max_stagnant = params.get('stagnation_limit_small', 100)

    start_time = time.time()
    iter_count = 0
    stagnant_count = 0
    restarts = 0

    print(f"  Initial Local Opt: {best.total_dist:.2f} (Gap: {calculate_gap(best.total_dist, instance.optimal):.2f}%)")

    if best.total_dist <= instance.optimal:
        print("  >> Optimal found immediately!")
        return best, history, {'iterations': 0, 'reheats': 0, 'early_stopped': True}
    
    while(time.time() - start_time) < max_time and iter_count < max_iter:
        iter_count += 1
        accepted_move = 0

        strength = params.get('perturb_strength_weak', 4)
        if stagnant_count > max_stagnant // 2:
            strength = params.get('perturb_strength_strong', 6)

        candidate = perturb_solution(curr, strength= strength)
        candidate = two_opt(candidate)

        delta = candidate.total_dist - curr.total_dist

        if accept_criterion(delta, T):
            curr = candidate
            accepted_move = 1
            if curr.total_dist < best.total_dist:
                best = curr.copy()
                gap = calculate_gap(best.total_dist, instance.optimal)
                print(f"  >> New Best: {best.total_dist:.0f} (Gap: {gap:.2f}%)")
            else:
                stagnant_count += 1

            if track:
                history['total_dist'].append(best.total_dist)
                history['current_dist'].append(curr.total_dist)
                history['temp'].append(T)
                history['accepted'].append(accepted_move)

            if stagnant_count >= max_stagnant:
                if restarts >= params.get('max_restarts_without_improvement', 5):
                    print(f"  >> Max restarts reached ({restarts}). Stopping.")
                    break

                print(f"  >> Stagnation ({max_stagnant}). RESTARTING.")
                curr = two_opt(perturb_solution(best, strength=params.get('perturb_strength_restart', 8)))
                T = params['T0_factor'] * instance.matrix_mean
                stagnant_count = 0
                restarts += 1
                if track: history['reheat_iters'].append(iter_count)

        T *= alpha
        if T < params.get('Tmin_factor', 0.001): 
            T = params['T0_factor'] * instance.matrix_mean * 0.5
    
    stats = {'iterations': iter_count, 'restarts': restarts, 'early_stopped': False}
    print(f"\nOptimization Complete: Gap {calculate_gap(best.total_dist, instance.optimal):.2f}% | Time {format_time(time.time() - start_time)}")
    return best, history, stats
print("Enhanced SA Ready! ✅")

Enhanced SA Ready! ✅
